In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = Path.cwd().parent  
PARQUET_PATH = BASE_PATH / "data" / "mv_dataset_parquet"
master_bookings = pd.read_parquet(PARQUET_PATH / "bookings.parquet")

Drop Duplicates - 101,545 found

In [2]:
before_rows = len(master_bookings)

master_bookings.drop_duplicates(inplace=True)

after_rows = len(master_bookings)

print(f"Rows before: {before_rows}")
print(f"Rows after:  {after_rows}")
print(f"Duplicates removed: {before_rows - after_rows}")

Rows before: 450930
Rows after:  349385
Duplicates removed: 101545


Calculate Missing

In [3]:
# Calculate missing data percentage for each column
missing_stats = pd.DataFrame({
    'missing_count': master_bookings.isnull().sum(),
    'missing_percentage': (master_bookings.isnull().sum() / len(master_bookings)) * 100
})

# Keep only columns with missing values
missing_stats = missing_stats[missing_stats['missing_count'] > 0]

# Sort by missing percentage
missing_stats = missing_stats.sort_values('missing_percentage', ascending=False)

print("\nMissing Data Summary (Non-Zero Only):")
print(missing_stats)


Missing Data Summary (Non-Zero Only):
                            missing_count  missing_percentage
deleted_at                         349385          100.000000
voucher_id                         349385          100.000000
guest_id                           349233           99.956495
refundable_amount_cents            348465           99.736680
refund_guarantee_fee_cents         348465           99.736680
voucher_added_at                   318938           91.285545
email                              179253           51.305294
phone                              178534           51.099503
user_id                            137909           39.471929
medium                               1202            0.344033
channel_name                           15            0.004293
date                                    1            0.000286
user_email                              1            0.000286
username                                1            0.000286


Booking Date Formatting

In [4]:
master_bookings['booking_date'] = pd.to_datetime(master_bookings['date'], errors='coerce')
master_bookings['day_of_week_index'] = pd.to_datetime(master_bookings['booking_date']).dt.weekday
master_bookings['day_of_week'] = pd.to_datetime(master_bookings['booking_date']).dt.day_name()

In [5]:
master_bookings.columns

Index(['updated_at_non_null', 'ack', 'active', 'arrived', 'channel',
       'created_at', 'created_incremental_refresh', 'date', 'deleted_at',
       'email', 'for_locking_system', 'guest_id', 'id', 'is_order_now',
       'is_temporary', 'is_valid_reservation', 'medium', 'no_show',
       'party_size', 'phone', 'restaurant_id', 'total_price_v2_cents',
       'updated_at', 'used_voucher_amount_by_hh_cents',
       'used_voucher_amount_by_restaurant_cents', 'user_id',
       'voucher_added_at', 'voucher_id', 'restaurant_name', 'hh_package_type',
       'package_price', 'revenue', 'channel_name', 'user_email', 'username',
       'city_name', 'user_type', 'has_refund_guarantee',
       'refund_guarantee_status', 'refundable_amount_cents',
       'refund_guarantee_fee_cents', 'payment_type', 'booking_date',
       'day_of_week_index', 'day_of_week'],
      dtype='object')

In [6]:
# -----------------------------------
# Initialize outlier flags
# -----------------------------------
master_bookings['is_outlier'] = False
master_bookings['outlier_reason'] = ''

# -----------------------------------
# Party size outliers
# -----------------------------------
party_size_outliers = (master_bookings['party_size'] < 1) | (master_bookings['party_size'] > 50)
master_bookings.loc[party_size_outliers, 'is_outlier'] = True
master_bookings.loc[party_size_outliers, 'outlier_reason'] += 'party_size_invalid; '


In [7]:
    
# -----------------------------------
# Revenue outliers
# Adjust threshold if your business has high-value package bookings
# -----------------------------------
revenue_outliers = (master_bookings['revenue'] < 0) | (master_bookings['revenue'] > 10000)
master_bookings.loc[revenue_outliers, 'is_outlier'] = True
master_bookings.loc[revenue_outliers, 'outlier_reason'] += 'revenue_invalid; '

# -----------------------------------
# Package price outliers
# -----------------------------------
package_price_outliers = (master_bookings['package_price'] < 0) | (master_bookings['package_price'] > 5000)
master_bookings.loc[package_price_outliers, 'is_outlier'] = True
master_bookings.loc[package_price_outliers, 'outlier_reason'] += 'package_price_invalid; '

# -----------------------------------
# Voucher amount outliers
# -----------------------------------
hh_voucher_outliers = (
    (master_bookings['used_voucher_amount_by_hh_cents'] < 0) |
    (master_bookings['used_voucher_amount_by_hh_cents'] > 1_000_000)
)
master_bookings.loc[hh_voucher_outliers, 'is_outlier'] = True
master_bookings.loc[hh_voucher_outliers, 'outlier_reason'] += 'hh_voucher_invalid; '

restaurant_voucher_outliers = (
    (master_bookings['used_voucher_amount_by_restaurant_cents'] < 0) |
    (master_bookings['used_voucher_amount_by_restaurant_cents'] > 1_000_000)
)
master_bookings.loc[restaurant_voucher_outliers, 'is_outlier'] = True
master_bookings.loc[restaurant_voucher_outliers, 'outlier_reason'] += 'restaurant_voucher_invalid; '

# -----------------------------------
# Refund amount / fee outliers
# -----------------------------------
refundable_amount_outliers = (
    (master_bookings['refundable_amount_cents'] < 0) |
    (master_bookings['refundable_amount_cents'] > 1_000_000)
)
master_bookings.loc[refundable_amount_outliers, 'is_outlier'] = True
master_bookings.loc[refundable_amount_outliers, 'outlier_reason'] += 'refundable_amount_invalid; '

refund_fee_outliers = (
    (master_bookings['refund_guarantee_fee_cents'] < 0) |
    (master_bookings['refund_guarantee_fee_cents'] > 100_000)
)
master_bookings.loc[refund_fee_outliers, 'is_outlier'] = True
master_bookings.loc[refund_fee_outliers, 'outlier_reason'] += 'refund_fee_invalid; '

# -----------------------------------
# Booking status contradictions
# -----------------------------------
arrived_and_no_show_outliers = (master_bookings['arrived'] == True) & (master_bookings['no_show'] == True)
master_bookings.loc[arrived_and_no_show_outliers, 'is_outlier'] = True
master_bookings.loc[arrived_and_no_show_outliers, 'outlier_reason'] += 'arrived_and_no_show; '

active_and_deleted_outliers = (master_bookings['active'] == True) & (master_bookings['deleted_at'].notna())
master_bookings.loc[active_and_deleted_outliers, 'is_outlier'] = True
master_bookings.loc[active_and_deleted_outliers, 'outlier_reason'] += 'active_but_deleted; '

voucher_missing_id_outliers = (
    (
        master_bookings['used_voucher_amount_by_hh_cents'].fillna(0) > 0
    ) | (
        master_bookings['used_voucher_amount_by_restaurant_cents'].fillna(0) > 0
    )
) & (master_bookings['voucher_id'].isna())
master_bookings.loc[voucher_missing_id_outliers, 'is_outlier'] = True
master_bookings.loc[voucher_missing_id_outliers, 'outlier_reason'] += 'voucher_amount_but_no_voucher_id; '

# -----------------------------------
# Date outliers
# -----------------------------------
created_after_booking_outliers = pd.to_datetime(master_bookings['created_at'], errors='coerce') > pd.to_datetime(master_bookings['date'], errors='coerce')
master_bookings.loc[created_after_booking_outliers, 'is_outlier'] = True
master_bookings.loc[created_after_booking_outliers, 'outlier_reason'] += 'created_after_booking_date; '

updated_before_created_outliers = pd.to_datetime(master_bookings['updated_at'], errors='coerce') < pd.to_datetime(master_bookings['created_at'], errors='coerce')
master_bookings.loc[updated_before_created_outliers, 'is_outlier'] = True
master_bookings.loc[updated_before_created_outliers, 'outlier_reason'] += 'updated_before_created; '

# -----------------------------------
# Summary
# -----------------------------------
print("=" * 60)
print("OUTLIER DETECTION SUMMARY")
print("=" * 60)
print(f"Total records: {len(master_bookings):,}")
print(f"Total outliers: {master_bookings['is_outlier'].sum():,} ({master_bookings['is_outlier'].sum()/len(master_bookings)*100:.2f}%)")
print()

print("Outlier breakdown:")
print(f"  Party size outliers (< 1 or > 50): {party_size_outliers.sum():,}")
print(f"  Revenue outliers (< 0 or > 10000): {revenue_outliers.sum():,}")
print(f"  Package price outliers (< 0 or > 5000): {package_price_outliers.sum():,}")
print(f"  HH voucher amount outliers: {hh_voucher_outliers.sum():,}")
print(f"  Restaurant voucher amount outliers: {restaurant_voucher_outliers.sum():,}")
print(f"  Refundable amount outliers: {refundable_amount_outliers.sum():,}")
print(f"  Refund fee outliers: {refund_fee_outliers.sum():,}")
print(f"  Arrived + no_show contradictions: {arrived_and_no_show_outliers.sum():,}")
print(f"  Active but deleted contradictions: {active_and_deleted_outliers.sum():,}")
print(f"  Voucher amount but no voucher_id: {voucher_missing_id_outliers.sum():,}")
print(f"  Created after booking date: {created_after_booking_outliers.sum():,}")
print(f"  Updated before created: {updated_before_created_outliers.sum():,}")
print()

# Optional: inspect flagged rows
outliers_df = master_bookings[master_bookings['is_outlier']].copy()
print(outliers_df[['id', 'restaurant_name', 'date', 'party_size', 'revenue', 'outlier_reason']].head(20))

OUTLIER DETECTION SUMMARY
Total records: 349,385
Total outliers: 349,335 (99.99%)

Outlier breakdown:
  Party size outliers (< 1 or > 50): 39
  Revenue outliers (< 0 or > 10000): 303,101
  Package price outliers (< 0 or > 5000): 349,289
  HH voucher amount outliers: 19
  Restaurant voucher amount outliers: 0
  Refundable amount outliers: 54
  Refund fee outliers: 8
  Arrived + no_show contradictions: 0
  Active but deleted contradictions: 0
  Voucher amount but no voucher_id: 30,422
  Created after booking date: 148,334
  Updated before created: 338

         id                             restaurant_name        date  \
0   7502901           Praram 9 Kaiyang The Jas Ramindra  2025-05-18   
1   7496395       The Dining Room at Grand Hyatt Erawan  2025-05-18   
2   7497808       The Dining Room at Grand Hyatt Erawan  2025-05-18   
3   7497891       The Dining Room at Grand Hyatt Erawan  2025-05-18   
4   7502196          Oishi Eaterium Future Park Rangsit  2025-05-18   
5   7502442      

,updated_at_non_null,ack,active,arrived,channel,created_at,created_incremental_refresh,date,deleted_at,email,...,has_refund_guarantee,refund_guarantee_status,refundable_amount_cents,refund_guarantee_fee_cents,payment_type,booking_date,day_of_week_index,day_of_week,is_outlier,outlier_reason
0,2025-05-19 15:09:22,True,True,True,8,2025-05-18 06:54:33,2025-05-18 06:54:33,2025-05-18,NaN,None,...,1,pending,NaN,NaN,credit card,2025-05-18,6.0,Sunday,True,revenue_invalid; package_price_invalid; vouche...
1,2025-05-19 11:00:35,True,True,False,1137,2025-05-15 12:47:17,2025-05-15 12:47:17,2025-05-18,NaN,None,...,1,pending,NaN,NaN,pay at store,2025-05-18,6.0,Sunday,True,revenue_invalid; package_price_invalid;
2,2025-05-16 06:27:26,True,True,False,8,2025-05-16 06:26:38,2025-05-16 06:26:38,2025-05-18,NaN,pimnava.k@gmail.com,...,1,pending,NaN,NaN,pay at store,2025-05-18,6.0,Sunday,True,revenue_invalid; package_price_invalid;
3,2025-05-19 11:00:35,True,True,False,10,2025-05-16 07:03:30,2025-05-16 07:03:30,2025-05-18,NaN,None,...,1,pending,NaN,NaN,pay at store,2025-05-18,6.0,Sunday,True,revenue_invalid; package_price_invalid;
4,2025-05-19 04:00:17,True,True,True,8,2025-05-18 02:09:22,2025-05-18 02:09:22,2025-05-18,NaN,None,...,1,pending,NaN,NaN,pay at store,2025-05-18,6.0,Sunday,True,revenue_invalid; package_price_invalid; create...


In [11]:
master_bookings.to_parquet(PARQUET_PATH / "bookings_cleaned.parquet", index=False)